# **Prepocessing Data**

In [1]:
!pip install thefuzz python-Levenshtein Sastrawi

import re
import nltk
import kagglehub
import pandas as pd
from thefuzz import fuzz
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## **Data Cleaning**


Load dataset Kaggle & filter provinsi Jawa Timur

In [2]:
path = kagglehub.dataset_download("sagitasantia/indonesia-tourism")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'indonesia-tourism' dataset.
Path to dataset files: /kaggle/input/indonesia-tourism


In [3]:
path = '/kaggle/input/indonesia-tourism/wisata_indonesia_new.csv'
df = pd.read_csv(path)
df.head()

,kategori,nama_wisata,latitude,longitude,alamat,provinsi,kota_kabupaten,nama_lengkap,path_gambar,deskripsi_bersih,Image_Path
0,air terjun,Air Terjun Aling-Aling,-8.176792,115.106284,"Air Terjun Aling-Aling, Panji Anom, Buleleng, ...",Bali,Buleleng,"Air Terjun Aling-Aling, Buleleng, Bali",gambar_wisata/Air_Terjun_Aling-Aling,Deskripsi tidak ditemukan,https://ksmtour.com/media/images/articles6/air...
1,air terjun,Air Terjun Bantimurung,-5.016520,119.685520,"Air Terjun Bantimurung, Jalan Poros Bantimurun...",Sulawesi Selatan,Sulawesi,"Air Terjun Bantimurung, Sulawesi, Sulawesi Sel...",gambar_wisata/Air_Terjun_Bantimurung,Air Terjun Bantimurung adalah salah satu air t...,https://cdn-2.tstatic.net/makassar/foto/bank/i...
2,air terjun,Air Terjun Benang Kelambu,-8.532770,116.337011,"Air Terjun Benang Kelambu, Pemotoh, Lombok Ten...",Nusa Tenggara Barat,Lombok Tengah,"Air Terjun Benang Kelambu, Lombok Tengah, Nusa...",gambar_wisata/Air_Terjun_Benang_Kelambu,Deskripsi tidak ditemukan,https://cdn.idntimes.com/content-images/commun...
3,air terjun,Air Terjun Benang Stokel,-8.533036,116.341379,"Air Terjun Benang Stokel, Pemotoh, Lombok Teng...",Nusa Tenggara Barat,Lombok Tengah,"Air Terjun Benang Stokel, Lombok Tengah, Nusa ...",gambar_wisata/Air_Terjun_Benang_Stokel,Deskripsi tidak ditemukan,https://s-light.tiket.photos/t/01E25EBZS3W0FY9...
4,air terjun,Air Terjun Bidadari,-2.928361,107.839043,"Air Terjun Bidadari, Nyuruk, Belitung Timur, K...",Kepulauan Bangka Belitung,Belitung Timur,"Air Terjun Bidadari, Belitung Timur, Kepulauan...",gambar_wisata/Air_Terjun_Bidadari,Daftar ini tidak lengkap dan hanya dapat dijad...,https://assets.pikiran-rakyat.com/crop/0x0:0x0...


In [4]:
jatim = df[df['provinsi'] == 'Jawa Timur']
jatim

,kategori,nama_wisata,latitude,longitude,alamat,provinsi,kota_kabupaten,nama_lengkap,path_gambar,deskripsi_bersih,Image_Path
5,air terjun,Air Terjun Coban Rondo,-7.884733,112.476970,"Coban Rondo, Kabupaten Malang, Jawa Timur, Jaw...",Jawa Timur,Kabupaten Malang,"Air Terjun Coban Rondo, Kabupaten Malang, Jawa...",gambar_wisata/Air_Terjun_Coban_Rondo,Air Terjun Coban Rondo merupakan air terjun ya...,https://www.kangamir.com/wp-content/uploads/20...
6,air terjun,Air Terjun Coban Talun,-7.802039,112.517049,"Air Terjun Coban Talun, Kota Batu, Jawa Timur,...",Jawa Timur,Kota Batu,"Air Terjun Coban Talun, Kota Batu, Jawa Timur",gambar_wisata/Air_Terjun_Coban_Talun,Deskripsi tidak ditemukan,https://travelspromo.com/wp-content/uploads/20...
8,air terjun,Air Terjun Dolo,-7.870637,111.835440,"Air Terjun Dolo, Mojo, Kabupaten Kediri, Jawa ...",Jawa Timur,Kabupaten Kediri,"Air Terjun Dolo, Kabupaten Kediri, Jawa Timur",gambar_wisata/Air_Terjun_Dolo,Air Terjun Dolo adalah salah satu tempat wisat...,https://asset.kompas.com/crops/vC88pQEIC8sQgBf...
20,air terjun,Air Terjun Tumpak Sewu,-8.238931,112.921426,"Kawasan Air Terjun Tumpak Sewu, Jagalan, Sidor...",Jawa Timur,Kabupaten Malang,"Air Terjun Tumpak Sewu, Kabupaten Malang, Jawa...",gambar_wisata/Air_Terjun_Tumpak_Sewu,Air Terjun Tumpak Sewu atau disebut juga Coban...,https://asset.kompas.com/crops/uJEyzMSHoxlQgXH...
23,alun-alun,Alun-Alun Banyuwangi,-8.207377,114.375929,"Jalan Tawang Alun, Biskalan, Patemon Wetan, Ba...",Jawa Timur,Banyuwangi,"Alun-Alun Banyuwangi, Banyuwangi, Jawa Timur",gambar_wisata/Alun-Alun_Banyuwangi,Banyuwangi (atau juga dikenal: Banyuwangi Kota...,https://cdn.idntimes.com/content-images/commun...
...,...,...,...,...,...,...,...,...,...,...,...
996,wisata tematik,Kampung Warna Warni Jodipan,-7.983122,112.637863,"Kampung Warna Warni Jodipan, Jalan Insinyur Ha...",Jawa Timur,Kota Malang,"Kampung Warna Warni Jodipan, Kota Malang, Jawa...",gambar_wisata/Kampung_Warna_Warni_Jodipan,"Kota Malang (bahasa Jawa: Hanacaraka: , Pegon...",https://asset.kompas.com/crops/AQnTDl9JE_k28of...
997,wisata tematik,Kota Tua Jakarta,-6.131183,106.810749,"Kota Tua Jakarta, Jalan Mangga Dua Raya, RW 04...",Jawa Timur,Daerah Khusus ibukota Jakarta,"Kota Tua Jakarta, Daerah Khusus ibukota Jakart...",gambar_wisata/Kota_Tua_Jakarta,Monumen Nasional yang disingkat dengan Monas a...,https://cdn.idntimes.com/content-images/post/2...
1001,wisata tematik,Taman Mini Indonesia Indah Jakarta,-6.302634,106.895296,"Taman Mini Indonesia Indah, Gang An Nur 2C, RW...",Jawa Timur,Jakarta Timur,"Taman Mini Indonesia Indah Jakarta, Jakarta Ti...",gambar_wisata/Taman_Mini_Indonesia_Indah_Jakarta,"Kabupaten Batang (bahasa Jawa: Hanacaraka: , ...",https://anekatempatwisata.com/wp-content/uploa...
1002,wisata tematik,Taman Safari Prigen,-7.752837,112.667453,"Taman Safari Indonesia Prigen, Taman Safari, J...",Jawa Timur,Pasuruan,"Taman Safari Prigen, Pasuruan, Jawa Timur",gambar_wisata/Taman_Safari_Prigen,The Grand Taman Safari Prigen Jawa Timur adala...,https://asset-2.tstatic.net/travel/foto/bank/i...


In [5]:
jatim['kota_kabupaten'].unique()

array(['Kabupaten Malang', 'Kota Batu', 'Kabupaten Kediri', 'Banyuwangi',
       'Blitar', 'Bojonegoro', 'Bondowoso', 'Gresik', 'Jember',
       'Kota Kediri', 'Kota Malang', 'Lamongan', 'Lumajang',
       'Kota Madiun', 'Kota Mojokerto', 'Ngawi', 'Pacitan',
       'Kota Pasuruan', 'Ponorogo', 'Kota Probolinggo', 'Serang ',
       'Sidoarjo', 'Situbondo', 'Surabaya', 'Trenggalek', 'Pasuruan',
       'Bangkalan', 'Trowulan', 'Kejagan', 'Mojokerto', 'Probolinggo',
       'Jakarta Timur', 'Daerah Khusus ibukota Jakarta',
       'Kabupaten Tangerang', 'Rangkasbitung', 'Pandeglang',
       'Tulungagung', 'Lebak', 'Mojopanggung', 'Tuban'], dtype=object)

Buang baris kota yang jelas bukan Jatim

In [6]:
jatim[jatim['kota_kabupaten'].isin(['Jakarta Timur', 'Daerah Khusus ibukota Jakarta', 'Serang ',
    'Kabupaten Tangerang', 'Rangkasbitung', 'Pandeglang', 'Lebak'])]

,kategori,nama_wisata,latitude,longitude,alamat,provinsi,kota_kabupaten,nama_lengkap,path_gambar,deskripsi_bersih,Image_Path
64,alun-alun,Alun-Alun Serang,-6.117348,106.153064,"Alun-Alun Serang, Kotabaru, Serang, Serang , J...",Jawa Timur,Serang,"Alun-Alun Serang, Serang á®á®¦á®á®, nan",gambar_wisata/Alun-Alun_Serang,Deskripsi tidak ditemukan,https://www.mitrabantennews.com/wp-content/upl...
197,gunung,Gunung Krakatau,-6.199619,106.926001,"Jalan Gunung Krakatau V, RW 05, Jatinegara, Ca...",Jawa Timur,Jakarta Timur,"Gunung Krakatau, Jakarta Timur, nan",gambar_wisata/Gunung_Krakatau,Gunung Pasaman adalah gunung yang terletak di ...,https://touaregadventure.com/wp-content/upload...
269,mall,Baywalk Mall Pluit,-6.107913,106.779619,"Baywalk Mall, Jalan Mandala Bahari, RW 10, Plu...",Jawa Timur,Daerah Khusus ibukota Jakarta,"Baywalk Mall Pluit, Daerah Khusus ibukota Jaka...",gambar_wisata/Baywalk_Mall_Pluit,Deskripsi tidak ditemukan,https://asset-2.tstatic.net/jakarta/foto/bank/...
271,mall,Central Park Mall,-6.177318,106.791497,"Central Park Mall, Kav.28, Madison Park Apartm...",Jawa Timur,Daerah Khusus ibukota Jakarta,"Central Park Mall, Daerah Khusus ibukota Jakar...",gambar_wisata/Central_Park_Mall,"Australia, dengan nama resmi Persemakmuran Aus...",https://i.pinimg.com/736x/03/89/7f/03897f717c6...
278,mall,Emporium Pluit Mall,-6.127579,106.790964,"Emporium Pluit Mall, Jalan Pluit Selatan Raya,...",Jawa Timur,Daerah Khusus ibukota Jakarta,"Emporium Pluit Mall, Daerah Khusus ibukota Jak...",gambar_wisata/Emporium_Pluit_Mall,Deskripsi tidak ditemukan,https://media-cdn.tripadvisor.com/media/photo-...
...,...,...,...,...,...,...,...,...,...,...,...
984,wisata religi,Vihara Dharma Bhakti Jakarta,-6.144868,106.780557,"Vihara Dharma Bhakti, Jalan Kusuma, RW 09, Wij...",Jawa Timur,Daerah Khusus ibukota Jakarta,"Vihara Dharma Bhakti Jakarta, Daerah Khusus ib...",gambar_wisata/Vihara_Dharma_Bhakti_Jakarta,"Aceh adalah sebuah provinsi di Pulau Sumatra, ...",http://www.jakartawalkingtour.com/wp-content/u...
985,wisata religi,Vihara Ekayana Arama Jakarta,-6.177717,106.778433,"Vihara Ekayana Arama, Jalan Mangga II, RW 08, ...",Jawa Timur,Daerah Khusus ibukota Jakarta,"Vihara Ekayana Arama Jakarta, Daerah Khusus ib...",gambar_wisata/Vihara_Ekayana_Arama_Jakarta,Deskripsi tidak ditemukan,https://buddhayana.or.id/assets/vihara/0208202...
997,wisata tematik,Kota Tua Jakarta,-6.131183,106.810749,"Kota Tua Jakarta, Jalan Mangga Dua Raya, RW 04...",Jawa Timur,Daerah Khusus ibukota Jakarta,"Kota Tua Jakarta, Daerah Khusus ibukota Jakart...",gambar_wisata/Kota_Tua_Jakarta,Monumen Nasional yang disingkat dengan Monas a...,https://cdn.idntimes.com/content-images/post/2...
1001,wisata tematik,Taman Mini Indonesia Indah Jakarta,-6.302634,106.895296,"Taman Mini Indonesia Indah, Gang An Nur 2C, RW...",Jawa Timur,Jakarta Timur,"Taman Mini Indonesia Indah Jakarta, Jakarta Ti...",gambar_wisata/Taman_Mini_Indonesia_Indah_Jakarta,"Kabupaten Batang (bahasa Jawa: Hanacaraka: , ...",https://anekatempatwisata.com/wp-content/uploa...


In [7]:
jatim2 = jatim[~jatim['kota_kabupaten'].isin(['Jakarta Timur', 'Daerah Khusus ibukota Jakarta', 'Serang ',
    'Kabupaten Tangerang', 'Rangkasbitung', 'Pandeglang', 'Lebak'])].copy()
jatim2

,kategori,nama_wisata,latitude,longitude,alamat,provinsi,kota_kabupaten,nama_lengkap,path_gambar,deskripsi_bersih,Image_Path
5,air terjun,Air Terjun Coban Rondo,-7.884733,112.476970,"Coban Rondo, Kabupaten Malang, Jawa Timur, Jaw...",Jawa Timur,Kabupaten Malang,"Air Terjun Coban Rondo, Kabupaten Malang, Jawa...",gambar_wisata/Air_Terjun_Coban_Rondo,Air Terjun Coban Rondo merupakan air terjun ya...,https://www.kangamir.com/wp-content/uploads/20...
6,air terjun,Air Terjun Coban Talun,-7.802039,112.517049,"Air Terjun Coban Talun, Kota Batu, Jawa Timur,...",Jawa Timur,Kota Batu,"Air Terjun Coban Talun, Kota Batu, Jawa Timur",gambar_wisata/Air_Terjun_Coban_Talun,Deskripsi tidak ditemukan,https://travelspromo.com/wp-content/uploads/20...
8,air terjun,Air Terjun Dolo,-7.870637,111.835440,"Air Terjun Dolo, Mojo, Kabupaten Kediri, Jawa ...",Jawa Timur,Kabupaten Kediri,"Air Terjun Dolo, Kabupaten Kediri, Jawa Timur",gambar_wisata/Air_Terjun_Dolo,Air Terjun Dolo adalah salah satu tempat wisat...,https://asset.kompas.com/crops/vC88pQEIC8sQgBf...
20,air terjun,Air Terjun Tumpak Sewu,-8.238931,112.921426,"Kawasan Air Terjun Tumpak Sewu, Jagalan, Sidor...",Jawa Timur,Kabupaten Malang,"Air Terjun Tumpak Sewu, Kabupaten Malang, Jawa...",gambar_wisata/Air_Terjun_Tumpak_Sewu,Air Terjun Tumpak Sewu atau disebut juga Coban...,https://asset.kompas.com/crops/uJEyzMSHoxlQgXH...
23,alun-alun,Alun-Alun Banyuwangi,-8.207377,114.375929,"Jalan Tawang Alun, Biskalan, Patemon Wetan, Ba...",Jawa Timur,Banyuwangi,"Alun-Alun Banyuwangi, Banyuwangi, Jawa Timur",gambar_wisata/Alun-Alun_Banyuwangi,Banyuwangi (atau juga dikenal: Banyuwangi Kota...,https://cdn.idntimes.com/content-images/commun...
...,...,...,...,...,...,...,...,...,...,...,...
986,wisata tematik,Batu Love Garden,-7.863866,112.542866,"Batu Love Garden, Pandanrejo, Kota Batu, Lowok...",Jawa Timur,Kota Batu,"Batu Love Garden, Kota Batu, Jawa Timur",gambar_wisata/Batu_Love_Garden,Artikel ini mengenai Pulau Batam. Lihat pula K...,https://asset-2.tstatic.net/travel/foto/bank/i...
992,wisata tematik,Eco Green Park Batu,-7.887662,112.526766,"Eco Green Park, Oro-oro Ombo, Kota Batu, Kloje...",Jawa Timur,Kota Batu,"Eco Green Park Batu, Kota Batu, Jawa Timur",gambar_wisata/Eco_Green_Park_Batu,Jawa Timur (atau juga diakronimkan sebagai Jat...,https://www.harapanrakyat.com/wp-content/uploa...
995,wisata tematik,House of Sampoerna,-7.230992,112.734146,"House of Sampoerna, 6, Jalan Sampoerna, RW 01,...",Jawa Timur,Surabaya,"House of Sampoerna, Surabaya, Jawa Timur",gambar_wisata/House_of_Sampoerna,PT Hanjaya Mandala Sampoerna Tbk (terkenal den...,https://www.lapispahlawan.co.id/uploads/6/2023...
996,wisata tematik,Kampung Warna Warni Jodipan,-7.983122,112.637863,"Kampung Warna Warni Jodipan, Jalan Insinyur Ha...",Jawa Timur,Kota Malang,"Kampung Warna Warni Jodipan, Kota Malang, Jawa...",gambar_wisata/Kampung_Warna_Warni_Jodipan,"Kota Malang (bahasa Jawa: Hanacaraka: , Pegon...",https://asset.kompas.com/crops/AQnTDl9JE_k28of...


In [8]:
jatim2['kota_kabupaten'].unique()

array(['Kabupaten Malang', 'Kota Batu', 'Kabupaten Kediri', 'Banyuwangi',
       'Blitar', 'Bojonegoro', 'Bondowoso', 'Gresik', 'Jember',
       'Kota Kediri', 'Kota Malang', 'Lamongan', 'Lumajang',
       'Kota Madiun', 'Kota Mojokerto', 'Ngawi', 'Pacitan',
       'Kota Pasuruan', 'Ponorogo', 'Kota Probolinggo', 'Sidoarjo',
       'Situbondo', 'Surabaya', 'Trenggalek', 'Pasuruan', 'Bangkalan',
       'Trowulan', 'Kejagan', 'Mojokerto', 'Probolinggo', 'Tulungagung',
       'Mojopanggung', 'Tuban'], dtype=object)

In [9]:
jatim2['kota_kabupaten'] = jatim2['kota_kabupaten'].replace({
    'Trowulan': 'Mojokerto',
    'Kejagan': 'Mojokerto',
})

Cek sinkronisasi nama wisata vs lokasi (fuzzy)

In [10]:
def cek_sinkronisasi(row):
    lokasi_lengkap = f"{row['alamat']} {row['kota_kabupaten']} {row['provinsi']}".lower()
    nama_wisata = str(row['nama_wisata']).lower()

    score = fuzz.token_set_ratio(nama_wisata, lokasi_lengkap)

    if score >= 70:
        return "Sesuai"
    elif score >= 40:
        return "Ragu-ragu"
    else:
        return "Tidak Cocok"

jatim2['status_validasi'] = jatim2.apply(cek_sinkronisasi, axis=1)

jatim2['skor_kemiripan'] = jatim2.apply(
    lambda x: fuzz.token_set_ratio(
        str(x['nama_wisata']).lower(),
        f"{x['alamat']} {x['kota_kabupaten']}".lower()
    ),
    axis=1
)

print(jatim2[jatim2['status_validasi'] == "Tidak Cocok"])

           kategori   nama_wisata  latitude   longitude  \
808  wisata edukasi  Jatim Park 1 -8.207928  114.356278   
809  wisata edukasi  Jatim Park 2 -8.207928  114.356278   

                                                alamat    provinsi  \
808  Kantor Maxim Indonesia - Banyuwangi, 1-2, Jala...  Jawa Timur   
809  Kantor Maxim Indonesia - Banyuwangi, 1-2, Jala...  Jawa Timur   

    kota_kabupaten                            nama_lengkap  \
808   Mojopanggung  Jatim Park 1, Mojopanggung, Jawa Timur   
809   Mojopanggung  Jatim Park 2, Mojopanggung, Jawa Timur   

                    path_gambar           deskripsi_bersih  \
808  gambar_wisata/Jatim_Park_1  Deskripsi tidak ditemukan   
809  gambar_wisata/Jatim_Park_2  Deskripsi tidak ditemukan   

                                            Image_Path status_validasi  \
808  https://wisato.id/wp-content/uploads/2020/03/f...     Tidak Cocok   
809  https://nahwatour.com/wp-content/uploads/2018/...     Tidak Cocok   

     skor_kemi

Perbaikan lokasi Jatim Park 1 & 2 (manual)

In [11]:
perbaikan = {
    'Jatim Park 1': {
        'kota': 'Kota Batu',
        'alamat': 'Jl. Kartini No. 2, Sisir, Kec. Batu',
        'nama_lengkap': 'Jatim Park 1, Kota Batu, Jawa Timur'
    },
    'Jatim Park 2': {
        'kota': 'Kota Batu',
        'alamat': 'Jl. Oro-Oro Ombo No. 9, Temas, Kec. Batu',
        'nama_lengkap': 'Jatim Park 2, Kota Batu, Jawa Timur'
    }
}

for nama, info in perbaikan.items():
    mask = jatim2['nama_wisata'] == nama
    jatim2.loc[mask, 'kota_kabupaten'] = info['kota']
    jatim2.loc[mask, 'alamat'] = info['alamat']
    jatim2.loc[mask, 'provinsi'] = 'Jawa Timur'

print("Proses perbaikan selesai!")
print(jatim2[jatim2['nama_wisata'].str.contains('Jatim Park')][['nama_wisata', 'alamat', 'kota_kabupaten']])

Proses perbaikan selesai!
      nama_wisata                                    alamat kota_kabupaten
808  Jatim Park 1       Jl. Kartini No. 2, Sisir, Kec. Batu      Kota Batu
809  Jatim Park 2  Jl. Oro-Oro Ombo No. 9, Temas, Kec. Batu      Kota Batu


In [12]:
jatim2.isnull().sum()

,0
kategori,0
nama_wisata,0
latitude,0
longitude,0
alamat,0
provinsi,0
kota_kabupaten,0
nama_lengkap,0
path_gambar,0
deskripsi_bersih,0


In [13]:
jumlah = (jatim2['deskripsi_bersih'] == "Deskripsi tidak ditemukan").sum()
print("Jumlah:", jumlah)

Jumlah: 12


Simpan untuk diedit manual di Excel

In [14]:
jatim2.to_csv('jatim2.csv', index=False, sep=';')

In [15]:
!ls -l /content/jatim2.csv

-rw-r--r-- 1 root root 69272 Mar  8 14:35 /content/jatim2.csv


In [16]:
df_saved = pd.read_csv('jatim2.csv', sep=';')
df_saved.head()

,kategori,nama_wisata,latitude,longitude,alamat,provinsi,kota_kabupaten,nama_lengkap,path_gambar,deskripsi_bersih,Image_Path,status_validasi,skor_kemiripan
0,air terjun,Air Terjun Coban Rondo,-7.884733,112.476970,"Coban Rondo, Kabupaten Malang, Jawa Timur, Jaw...",Jawa Timur,Kabupaten Malang,"Air Terjun Coban Rondo, Kabupaten Malang, Jawa...",gambar_wisata/Air_Terjun_Coban_Rondo,Air Terjun Coban Rondo merupakan air terjun ya...,https://www.kangamir.com/wp-content/uploads/20...,Ragu-ragu,67
1,air terjun,Air Terjun Coban Talun,-7.802039,112.517049,"Air Terjun Coban Talun, Kota Batu, Jawa Timur,...",Jawa Timur,Kota Batu,"Air Terjun Coban Talun, Kota Batu, Jawa Timur",gambar_wisata/Air_Terjun_Coban_Talun,Deskripsi tidak ditemukan,https://travelspromo.com/wp-content/uploads/20...,Sesuai,100
2,air terjun,Air Terjun Dolo,-7.870637,111.835440,"Air Terjun Dolo, Mojo, Kabupaten Kediri, Jawa ...",Jawa Timur,Kabupaten Kediri,"Air Terjun Dolo, Kabupaten Kediri, Jawa Timur",gambar_wisata/Air_Terjun_Dolo,Air Terjun Dolo adalah salah satu tempat wisat...,https://asset.kompas.com/crops/vC88pQEIC8sQgBf...,Sesuai,100
3,air terjun,Air Terjun Tumpak Sewu,-8.238931,112.921426,"Kawasan Air Terjun Tumpak Sewu, Jagalan, Sidor...",Jawa Timur,Kabupaten Malang,"Air Terjun Tumpak Sewu, Kabupaten Malang, Jawa...",gambar_wisata/Air_Terjun_Tumpak_Sewu,Air Terjun Tumpak Sewu atau disebut juga Coban...,https://asset.kompas.com/crops/uJEyzMSHoxlQgXH...,Sesuai,100
4,alun-alun,Alun-Alun Banyuwangi,-8.207377,114.375929,"Jalan Tawang Alun, Biskalan, Patemon Wetan, Ba...",Jawa Timur,Banyuwangi,"Alun-Alun Banyuwangi, Banyuwangi, Jawa Timur",gambar_wisata/Alun-Alun_Banyuwangi,Banyuwangi (atau juga dikenal: Banyuwangi Kota...,https://cdn.idntimes.com/content-images/commun...,Sesuai,100


In [17]:
jatim2[jatim2["deskripsi_bersih"] == "Deskripsi tidak ditemukan"]

,kategori,nama_wisata,latitude,longitude,alamat,provinsi,kota_kabupaten,nama_lengkap,path_gambar,deskripsi_bersih,Image_Path,status_validasi,skor_kemiripan
6,air terjun,Air Terjun Coban Talun,-7.802039,112.517049,"Air Terjun Coban Talun, Kota Batu, Jawa Timur,...",Jawa Timur,Kota Batu,"Air Terjun Coban Talun, Kota Batu, Jawa Timur",gambar_wisata/Air_Terjun_Coban_Talun,Deskripsi tidak ditemukan,https://travelspromo.com/wp-content/uploads/20...,Sesuai,100
103,bukit,Bukit Kuneer,-7.813390,112.633901,"Bukit Kuneer, Jalur Pendakian Gunung Arjuno vi...",Jawa Timur,Kabupaten Malang,"Bukit Kuneer, Kabupaten Malang, Jawa Timur",gambar_wisata/Bukit_Kuneer,Deskripsi tidak ditemukan,https://www.gresiksatu.com/wp-content/uploads/...,Sesuai,100
104,bukit,Bukit Kuneer Malang,-7.813390,112.633901,"Bukit Kuneer, Jalur Pendakian Gunung Arjuno vi...",Jawa Timur,Kabupaten Malang,"Bukit Kuneer Malang, Kabupaten Malang, Jawa Timur",gambar_wisata/Bukit_Kuneer_Malang,Deskripsi tidak ditemukan,https://www.gresiksatu.com/wp-content/uploads/...,Sesuai,100
132,candi,Candi Brahu,-7.542846,112.374398,"Candi Brahu, Jalan Raden Wijaya, Kejagan, Trow...",Jawa Timur,Mojokerto,"Candi Brahu, Kejagan, Jawa Timur",gambar_wisata/Candi_Brahu,Deskripsi tidak ditemukan,https://ksmtour.com/media/images/articles26/ca...,Sesuai,100
444,pantai,Pantai Bandealit,-8.481706,113.710736,"Pantai Bandealit, Kebonpantai, Jember, Jawa Ti...",Jawa Timur,Jember,"Pantai Bandealit, Jember, Jawa Timur",gambar_wisata/Pantai_Bandealit,Deskripsi tidak ditemukan,https://nativeindonesia.com/wp-content/uploads...,Sesuai,100
507,pantai,Pantai Pelang,-8.258722,111.424883,"Pantai Pelang, Trenggalek, Jawa Timur, Jawa, 6...",Jawa Timur,Trenggalek,"Pantai Pelang, Trenggalek, Jawa Timur",gambar_wisata/Pantai_Pelang,Deskripsi tidak ditemukan,https://ksmtour.com/media/images/articles26/Pa...,Sesuai,100
525,pantai,Pantai Soge,-8.247559,111.263429,"Pantai Soge, Pacitan, Jawa Timur, Jawa, 63572,...",Jawa Timur,Pacitan,"Pantai Soge, Pacitan, Jawa Timur",gambar_wisata/Pantai_Soge,Deskripsi tidak ditemukan,https://ksmtour.com/media/images/articles21/pa...,Sesuai,100
692,wisata alam,Goa Pandawa,-7.852614,112.501001,"Wisata Goa Pandawa, Sumberejo, Kota Batu, Jawa...",Jawa Timur,Kota Batu,"Goa Pandawa, Kota Batu, Jawa Timur",gambar_wisata/Goa_Pandawa,Deskripsi tidak ditemukan,https://4.bp.blogspot.com/-dKj-UmgQ2CQ/WdCnr70...,Sesuai,100
792,wisata alam,Suramadu Bridge,-7.184276,112.780303,"Jembatan Suramadu, Jalan Haji Mohammad Noer, R...",Jawa Timur,Bangkalan,"Suramadu Bridge, Bangkalan, Jawa Timur",gambar_wisata/Suramadu_Bridge,Deskripsi tidak ditemukan,https://www.nadiamurad.org/wp-content/uploads/...,Sesuai,70
808,wisata edukasi,Jatim Park 1,-8.207928,114.356278,"Jl. Kartini No. 2, Sisir, Kec. Batu",Jawa Timur,Kota Batu,"Jatim Park 1, Mojopanggung, Jawa Timur",gambar_wisata/Jatim_Park_1,Deskripsi tidak ditemukan,https://wisato.id/wp-content/uploads/2020/03/f...,Tidak Cocok,16


## **Data Cleaning Lanjutan**


Menggunakan file yang sudah diedit manual di excel

In [18]:
jatim3 = pd.read_csv('https://raw.githubusercontent.com/Qozuu/datamining-chatbot/main/jatim-asolole.csv', sep=';', encoding="utf-8-sig")
jatim3

,kategori,nama_wisata,latitude,longitude,alamat,provinsi,kota_kabupaten,nama_lengkap,path_gambar,deskripsi_bersih,Image_Path,status_validasi,skor_kemiripan,Column1,Column2
0,air terjun,Air Terjun Coban Rondo,-78847333,1124769702,"Coban Rondo, Kabupaten Malang, Jawa Timur, Jaw...",Jawa Timur,Kabupaten Malang,"Air Terjun Coban Rondo, Kabupaten Malang, Jawa...",gambar_wisata/Air_Terjun_Coban_Rondo,Air Terjun Coban Rondo merupakan salah satu ob...,https://www.kangamir.com/wp-content/uploads/20...,Ragu-ragu,67,NaN,NaN
1,air terjun,Air Terjun Coban Talun,-78020389,1125170485,"Air Terjun Coban Talun, Kota Batu, Jawa Timur,...",Jawa Timur,Kota Batu,"Air Terjun Coban Talun, Kota Batu, Jawa Timur",gambar_wisata/Air_Terjun_Coban_Talun,Air Terjun Coban Talun adalah air terjun setin...,https://travelspromo.com/wp-content/uploads/20...,Sesuai,100,NaN,NaN
2,air terjun,Air Terjun Dolo,-78706373,1118354398,"Air Terjun Dolo, Mojo, Kabupaten Kediri, Jawa ...",Jawa Timur,Kabupaten Kediri,"Air Terjun Dolo, Kabupaten Kediri, Jawa Timur",gambar_wisata/Air_Terjun_Dolo,Air Terjun Dolo adalah salah satu tempat wisat...,https://asset.kompas.com/crops/vC88pQEIC8sQgBf...,Sesuai,100,NaN,NaN
3,air terjun,Air Terjun Tumpak Sewu,-82389311,1129214262,"Kawasan Air Terjun Tumpak Sewu, Jagalan, Sidor...",Jawa Timur,Kabupaten Malang,"Air Terjun Tumpak Sewu, Kabupaten Malang, Jawa...",gambar_wisata/Air_Terjun_Tumpak_Sewu,Air Terjun Tumpak Sewu atau disebut juga Coban...,https://asset.kompas.com/crops/uJEyzMSHoxlQgXH...,Sesuai,100,NaN,NaN
4,alun-alun,Alun-Alun Banyuwangi,-82073769,1143759291,"Jalan Tawang Alun, Biskalan, Patemon Wetan, Ba...",Jawa Timur,Banyuwangi,"Alun-Alun Banyuwangi, Banyuwangi, Jawa Timur",gambar_wisata/Alun-Alun_Banyuwangi,lun-Alun Banyuwangi adalah ruang terbuka publi...,https://cdn.idntimes.com/content-images/commun...,Sesuai,100,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110,wisata religi,Pura Agung Blambangan,-84316781,1143200224,"Pura Agung Blambangan, Jalan Ngurah Rai, Tembo...",Jawa Timur,Banyuwangi,"Pura Agung Blambangan, Banyuwangi, Jawa Timur",gambar_wisata/Pura_Agung_Blambangan,Pura Agung Blambangan di Banyuwangi merupakan ...,https://cdn.statically.io/img/liburanmulu.com/...,Sesuai,100,NaN,NaN
111,wisata tematik,Batu Love Garden,-78638662,1125428658,"Batu Love Garden, Pandanrejo, Kota Batu, Lowok...",Jawa Timur,Kota Batu,"Batu Love Garden, Kota Batu, Jawa Timur",gambar_wisata/Batu_Love_Garden,Batu Love Garden (Baloga) di Kota Batu adalah ...,https://asset-2.tstatic.net/travel/foto/bank/i...,Sesuai,100,NaN,NaN
112,wisata tematik,House of Sampoerna,-72309923,1127341461,"House of Sampoerna, 6, Jalan Sampoerna, RW 01,...",Jawa Timur,Surabaya,"House of Sampoerna, Surabaya, Jawa Timur",gambar_wisata/House_of_Sampoerna,House of Sampoerna di kawasan Kota Tua Surabay...,https://www.lapispahlawan.co.id/uploads/6/2023...,Sesuai,100,NaN,NaN
113,wisata tematik,Kampung Warna Warni Jodipan,-79831221,1126378633,"Kampung Warna Warni Jodipan, Jalan Insinyur Ha...",Jawa Timur,Kota Malang,"Kampung Warna Warni Jodipan, Kota Malang, Jawa...",gambar_wisata/Kampung_Warna_Warni_Jodipan,Kampung Warna Warni Jodipan di bantaran Sungai...,https://asset.kompas.com/crops/AQnTDl9JE_k28of...,Sesuai,100,NaN,NaN


In [19]:
jumlah = (jatim3['deskripsi_bersih'] == "Deskripsi tidak ditemukan").sum()
print("Jumlah:", jumlah)

Jumlah: 0


## **Data Reduction**

Hapus kolom yang tidak dipakai chatbot

In [20]:
jatim3 = jatim3.drop(columns=['latitude', 'longitude', 'provinsi', 'path_gambar', 'Image_Path', 'status_validasi', 'skor_kemiripan', 'Column1', 'Column2'])
jatim3

,kategori,nama_wisata,alamat,kota_kabupaten,nama_lengkap,deskripsi_bersih
0,air terjun,Air Terjun Coban Rondo,"Coban Rondo, Kabupaten Malang, Jawa Timur, Jaw...",Kabupaten Malang,"Air Terjun Coban Rondo, Kabupaten Malang, Jawa...",Air Terjun Coban Rondo merupakan salah satu ob...
1,air terjun,Air Terjun Coban Talun,"Air Terjun Coban Talun, Kota Batu, Jawa Timur,...",Kota Batu,"Air Terjun Coban Talun, Kota Batu, Jawa Timur",Air Terjun Coban Talun adalah air terjun setin...
2,air terjun,Air Terjun Dolo,"Air Terjun Dolo, Mojo, Kabupaten Kediri, Jawa ...",Kabupaten Kediri,"Air Terjun Dolo, Kabupaten Kediri, Jawa Timur",Air Terjun Dolo adalah salah satu tempat wisat...
3,air terjun,Air Terjun Tumpak Sewu,"Kawasan Air Terjun Tumpak Sewu, Jagalan, Sidor...",Kabupaten Malang,"Air Terjun Tumpak Sewu, Kabupaten Malang, Jawa...",Air Terjun Tumpak Sewu atau disebut juga Coban...
4,alun-alun,Alun-Alun Banyuwangi,"Jalan Tawang Alun, Biskalan, Patemon Wetan, Ba...",Banyuwangi,"Alun-Alun Banyuwangi, Banyuwangi, Jawa Timur",lun-Alun Banyuwangi adalah ruang terbuka publi...
...,...,...,...,...,...,...
110,wisata religi,Pura Agung Blambangan,"Pura Agung Blambangan, Jalan Ngurah Rai, Tembo...",Banyuwangi,"Pura Agung Blambangan, Banyuwangi, Jawa Timur",Pura Agung Blambangan di Banyuwangi merupakan ...
111,wisata tematik,Batu Love Garden,"Batu Love Garden, Pandanrejo, Kota Batu, Lowok...",Kota Batu,"Batu Love Garden, Kota Batu, Jawa Timur",Batu Love Garden (Baloga) di Kota Batu adalah ...
112,wisata tematik,House of Sampoerna,"House of Sampoerna, 6, Jalan Sampoerna, RW 01,...",Surabaya,"House of Sampoerna, Surabaya, Jawa Timur",House of Sampoerna di kawasan Kota Tua Surabay...
113,wisata tematik,Kampung Warna Warni Jodipan,"Kampung Warna Warni Jodipan, Jalan Insinyur Ha...",Kota Malang,"Kampung Warna Warni Jodipan, Kota Malang, Jawa...",Kampung Warna Warni Jodipan di bantaran Sungai...


Gabung dengan `tambahan-chatbot.csv` (data tambahan yang diinput manual (wisata baru))

In [21]:
df = pd.read_csv('https://raw.githubusercontent.com/Qozuu/datamining-chatbot/main/tambahan-chatbot.csv', sep=';', encoding='latin1')
df

,kategori,nama_wisata,alamat,provinsi,kota_kabupaten,nama_lengkap,deskripsi_bersih
0,wisata alam,Ranu Kumbolo,"Ranu Kumbolo, Pasrujambe, Kabupaten Lumajang, ...",Jawa Timur,Kabupaten Lumajang,"Ranu Kumbolo, Kabupaten Lumajang, Jawa Timur",Ranu Kumbolo adalah danau alam di kawasan Tama...
1,wahana keluarga,Hawai Waterpark Malang,"Hawai Waterpark, Jl. Graha Kencana Raya, Banja...",Jawa Timur,Kabupaten Malang,"Hawai Waterpark Malang, Kabupaten Malang, Jawa...",Hawai Waterpark Malang merupakan taman wisata ...
2,wahana keluarga,Malang Night Paradise,"Malang Night Paradise, Graha Kencana Raya, Jl....",Jawa Timur,Kota Malang,"Malang Night Paradise, Kota Malang, Jawa Timur",Malang Night Paradise adalah taman hiburan mal...
3,wisata alam,Ranu Pani,"Ranu Pani, Desa Ranu Pani, Senduro, Kabupaten ...",Jawa Timur,Kabupaten Lumajang,"Ranu Pani, Kabupaten Lumajang, Jawa Timur",Ranu Pani adalah danau kecil di Desa Ranu Pani...
4,wisata alam,Ranu Regulo,"Ranu Regulo, Desa Ranu Pani, Senduro, Kabupate...",Jawa Timur,Kabupaten Lumajang,"Ranu Regulo, Kabupaten Lumajang, Jawa Timur",Ranu Regulo adalah danau pegunungan di sekitar...
...,...,...,...,...,...,...,...
111,wisata kota,Kampung Biru Arema,"Kelurahan Kiduldalem, Kecamatan Klojen, Kota M...",Jawa Timur,Kota Malang,Kampung Biru Arema Malang,Kampung Biru Arema merupakan kawasan wisata te...
112,wisata kota,Jembatan Suramadu,"Kelurahan Kenjeran, Kecamatan Bulak, Kota Sura...",Jawa Timur,Surabaya,Jembatan Suramadu Surabaya,Jembatan Suramadu merupakan jembatan terpanjan...
113,agrowisata,Agrowisata Belimbing Karangsari,"Kelurahan Karangsari, Kecamatan Sukorejo, Kota...",Jawa Timur,Kota Blitar,Agrowisata Belimbing Karangsari Blitar,Agrowisata Belimbing Karangsari menawarkan pen...
114,agrowisata,Kebun Raya Purwodadi,"Desa Purwodadi, Kecamatan Purwodadi, Kabupaten...",Jawa Timur,Pasuruan,Kebun Raya Purwodadi Pasuruan,Kebun Raya Purwodadi merupakan kebun botani ya...


In [22]:
df.drop(columns=['provinsi'], inplace=True)
df

,kategori,nama_wisata,alamat,kota_kabupaten,nama_lengkap,deskripsi_bersih
0,wisata alam,Ranu Kumbolo,"Ranu Kumbolo, Pasrujambe, Kabupaten Lumajang, ...",Kabupaten Lumajang,"Ranu Kumbolo, Kabupaten Lumajang, Jawa Timur",Ranu Kumbolo adalah danau alam di kawasan Tama...
1,wahana keluarga,Hawai Waterpark Malang,"Hawai Waterpark, Jl. Graha Kencana Raya, Banja...",Kabupaten Malang,"Hawai Waterpark Malang, Kabupaten Malang, Jawa...",Hawai Waterpark Malang merupakan taman wisata ...
2,wahana keluarga,Malang Night Paradise,"Malang Night Paradise, Graha Kencana Raya, Jl....",Kota Malang,"Malang Night Paradise, Kota Malang, Jawa Timur",Malang Night Paradise adalah taman hiburan mal...
3,wisata alam,Ranu Pani,"Ranu Pani, Desa Ranu Pani, Senduro, Kabupaten ...",Kabupaten Lumajang,"Ranu Pani, Kabupaten Lumajang, Jawa Timur",Ranu Pani adalah danau kecil di Desa Ranu Pani...
4,wisata alam,Ranu Regulo,"Ranu Regulo, Desa Ranu Pani, Senduro, Kabupate...",Kabupaten Lumajang,"Ranu Regulo, Kabupaten Lumajang, Jawa Timur",Ranu Regulo adalah danau pegunungan di sekitar...
...,...,...,...,...,...,...
111,wisata kota,Kampung Biru Arema,"Kelurahan Kiduldalem, Kecamatan Klojen, Kota M...",Kota Malang,Kampung Biru Arema Malang,Kampung Biru Arema merupakan kawasan wisata te...
112,wisata kota,Jembatan Suramadu,"Kelurahan Kenjeran, Kecamatan Bulak, Kota Sura...",Surabaya,Jembatan Suramadu Surabaya,Jembatan Suramadu merupakan jembatan terpanjan...
113,agrowisata,Agrowisata Belimbing Karangsari,"Kelurahan Karangsari, Kecamatan Sukorejo, Kota...",Kota Blitar,Agrowisata Belimbing Karangsari Blitar,Agrowisata Belimbing Karangsari menawarkan pen...
114,agrowisata,Kebun Raya Purwodadi,"Desa Purwodadi, Kecamatan Purwodadi, Kabupaten...",Pasuruan,Kebun Raya Purwodadi Pasuruan,Kebun Raya Purwodadi merupakan kebun botani ya...


## **Data Integration**

Gabung (concat) data utama + data tambahan

In [23]:
jj = pd.concat([jatim3, df], ignore_index=True)
jj

,kategori,nama_wisata,alamat,kota_kabupaten,nama_lengkap,deskripsi_bersih
0,air terjun,Air Terjun Coban Rondo,"Coban Rondo, Kabupaten Malang, Jawa Timur, Jaw...",Kabupaten Malang,"Air Terjun Coban Rondo, Kabupaten Malang, Jawa...",Air Terjun Coban Rondo merupakan salah satu ob...
1,air terjun,Air Terjun Coban Talun,"Air Terjun Coban Talun, Kota Batu, Jawa Timur,...",Kota Batu,"Air Terjun Coban Talun, Kota Batu, Jawa Timur",Air Terjun Coban Talun adalah air terjun setin...
2,air terjun,Air Terjun Dolo,"Air Terjun Dolo, Mojo, Kabupaten Kediri, Jawa ...",Kabupaten Kediri,"Air Terjun Dolo, Kabupaten Kediri, Jawa Timur",Air Terjun Dolo adalah salah satu tempat wisat...
3,air terjun,Air Terjun Tumpak Sewu,"Kawasan Air Terjun Tumpak Sewu, Jagalan, Sidor...",Kabupaten Malang,"Air Terjun Tumpak Sewu, Kabupaten Malang, Jawa...",Air Terjun Tumpak Sewu atau disebut juga Coban...
4,alun-alun,Alun-Alun Banyuwangi,"Jalan Tawang Alun, Biskalan, Patemon Wetan, Ba...",Banyuwangi,"Alun-Alun Banyuwangi, Banyuwangi, Jawa Timur",lun-Alun Banyuwangi adalah ruang terbuka publi...
...,...,...,...,...,...,...
226,wisata kota,Kampung Biru Arema,"Kelurahan Kiduldalem, Kecamatan Klojen, Kota M...",Kota Malang,Kampung Biru Arema Malang,Kampung Biru Arema merupakan kawasan wisata te...
227,wisata kota,Jembatan Suramadu,"Kelurahan Kenjeran, Kecamatan Bulak, Kota Sura...",Surabaya,Jembatan Suramadu Surabaya,Jembatan Suramadu merupakan jembatan terpanjan...
228,agrowisata,Agrowisata Belimbing Karangsari,"Kelurahan Karangsari, Kecamatan Sukorejo, Kota...",Kota Blitar,Agrowisata Belimbing Karangsari Blitar,Agrowisata Belimbing Karangsari menawarkan pen...
229,agrowisata,Kebun Raya Purwodadi,"Desa Purwodadi, Kecamatan Purwodadi, Kabupaten...",Pasuruan,Kebun Raya Purwodadi Pasuruan,Kebun Raya Purwodadi merupakan kebun botani ya...


In [24]:
duplikat = jj.duplicated(subset="nama_wisata")

print("Jumlah data duplikat:", duplikat.sum())

Jumlah data duplikat: 12


In [25]:
jj[jj.duplicated(subset="nama_wisata", keep=False)]

,kategori,nama_wisata,alamat,kota_kabupaten,nama_lengkap,deskripsi_bersih
11,alun-alun,Alun-Alun Kota Batu,"Parkir Alun-Alun Kota Batu, Jalan RA. Kartini,...",Kota Batu,"Alun-Alun Kota Batu, Kota Batu, Jawa Timur",Alun-Alun Kota Batu merupakan salah satu ikon ...
12,alun-alun,Alun-Alun Kota Malang,"Alun-Alun Malang, Kiduldalem, Kota Malang, Jaw...",Kota Malang,"Alun-Alun Kota Malang, Kota Malang, Jawa Timur",Alun-Alun Malang (khususnya Alun-Alun Merdeka)...
27,bukit,Bukit Jamur,"Bukit Jamur Gresik, Sukowati, Gresik, Jawa Tim...",Gresik,"Bukit Jamur, Gresik, Jawa Timur",Bukit Jamur berada di Kabupaten Gresik dan dik...
28,bukit,Bukit Kapur Arosbaya,"Bukit Kapur Arosbaya, Jalan K.H. Zainal Alimin...",Bangkalan,"Bukit Kapur Arosbaya, Bangkalan, Jawa Timur",Bukit Kapur Arosbaya merupakan bekas area pena...
36,candi,Candi Jawi,"Candi Jawi, Jalan Raya Prigen, Prigen, Pasurua...",Pasuruan,"Candi Jawi, Pasuruan, Jawa Timur",Candi Jawi adalah candi Hindu–Buddha peninggal...
37,candi,Candi Kidal,"Candi Kidal, Jalan Raya Kidal, Jeding, Kidal, ...",Kabupaten Malang,"Candi Kidal, Kabupaten Malang, Jawa Timur","Candi Kidal berada di Desa Kidal, Kabupaten Ma..."
57,monumen,Monumen Kapal Selam,"Monumen Kapal Selam, 39, Jalan Pemuda, RW 05, ...",Surabaya,"Monumen Kapal Selam, Surabaya, Jawa Timur",Monumen Kapal Selam (Monkasel) adalah museum m...
60,museum,Museum Majapahit,"Museum Majapahit Trowulan, Jalan Jayanegara 01...",Trowulan,"Museum Majapahit, Trowulan, Jawa Timur",Museum Majapahit atau Museum Trowulan adalah m...
94,wisata edukasi,Kebun Raya Purwodadi,"Kebun Raya Purwodadi, Jalan Raya Surabaya - Ma...",Pasuruan,"Kebun Raya Purwodadi, Pasuruan, Jawa Timur",Kebun Raya Purwodadi merupakan salah satu kebu...
112,wisata tematik,House of Sampoerna,"House of Sampoerna, 6, Jalan Sampoerna, RW 01,...",Surabaya,"House of Sampoerna, Surabaya, Jawa Timur",House of Sampoerna di kawasan Kota Tua Surabay...


Buang duplikat, simpan satu baris per nama_wisata

In [26]:
jj = jj.drop_duplicates(subset="nama_wisata")
print(jj.duplicated(subset="nama_wisata").sum())

0


In [27]:
jj

,kategori,nama_wisata,alamat,kota_kabupaten,nama_lengkap,deskripsi_bersih
0,air terjun,Air Terjun Coban Rondo,"Coban Rondo, Kabupaten Malang, Jawa Timur, Jaw...",Kabupaten Malang,"Air Terjun Coban Rondo, Kabupaten Malang, Jawa...",Air Terjun Coban Rondo merupakan salah satu ob...
1,air terjun,Air Terjun Coban Talun,"Air Terjun Coban Talun, Kota Batu, Jawa Timur,...",Kota Batu,"Air Terjun Coban Talun, Kota Batu, Jawa Timur",Air Terjun Coban Talun adalah air terjun setin...
2,air terjun,Air Terjun Dolo,"Air Terjun Dolo, Mojo, Kabupaten Kediri, Jawa ...",Kabupaten Kediri,"Air Terjun Dolo, Kabupaten Kediri, Jawa Timur",Air Terjun Dolo adalah salah satu tempat wisat...
3,air terjun,Air Terjun Tumpak Sewu,"Kawasan Air Terjun Tumpak Sewu, Jagalan, Sidor...",Kabupaten Malang,"Air Terjun Tumpak Sewu, Kabupaten Malang, Jawa...",Air Terjun Tumpak Sewu atau disebut juga Coban...
4,alun-alun,Alun-Alun Banyuwangi,"Jalan Tawang Alun, Biskalan, Patemon Wetan, Ba...",Banyuwangi,"Alun-Alun Banyuwangi, Banyuwangi, Jawa Timur",lun-Alun Banyuwangi adalah ruang terbuka publi...
...,...,...,...,...,...,...
223,wisata sejarah,Benteng Van den Bosch,"Desa Pelem, Kecamatan Ngawi, Kabupaten Ngawi, ...",Ngawi,Benteng Van den Bosch Ngawi,Benteng Van den Bosch merupakan peninggalan ko...
224,wisata sejarah,Istana Gebang,"Kelurahan Bendogerit, Kecamatan Sananwetan, Ko...",Kota Blitar,Istana Gebang Blitar,Istana Gebang merupakan rumah masa kecil Presi...
226,wisata kota,Kampung Biru Arema,"Kelurahan Kiduldalem, Kecamatan Klojen, Kota M...",Kota Malang,Kampung Biru Arema Malang,Kampung Biru Arema merupakan kawasan wisata te...
227,wisata kota,Jembatan Suramadu,"Kelurahan Kenjeran, Kecamatan Bulak, Kota Sura...",Surabaya,Jembatan Suramadu Surabaya,Jembatan Suramadu merupakan jembatan terpanjan...


## **Data Transformation**

### **Case Folding**

In [28]:
cols = ['nama_wisata','alamat','kota_kabupaten','nama_lengkap']

for col in cols:
    jj[col] = jj[col].str.lower()

jj['case_folding'] = jj['deskripsi_bersih'].str.lower()

# hapus angka dan tanda baca
jj['case_folding'] = jj['case_folding'].apply(lambda x: re.sub(r'[^a-zA-Z\s]', '', str(x)))

/tmp/ipykernel_11716/3884506531.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  jj[col] = jj[col].str.lower()
/tmp/ipykernel_11716/3884506531.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  jj['case_folding'] = jj['deskripsi_bersih'].str.lower()
/tmp/ipykernel_11716/3884506531.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-

In [29]:
jj

,kategori,nama_wisata,alamat,kota_kabupaten,nama_lengkap,deskripsi_bersih,case_folding
0,air terjun,air terjun coban rondo,"coban rondo, kabupaten malang, jawa timur, jaw...",kabupaten malang,"air terjun coban rondo, kabupaten malang, jawa...",Air Terjun Coban Rondo merupakan salah satu ob...,air terjun coban rondo merupakan salah satu ob...
1,air terjun,air terjun coban talun,"air terjun coban talun, kota batu, jawa timur,...",kota batu,"air terjun coban talun, kota batu, jawa timur",Air Terjun Coban Talun adalah air terjun setin...,air terjun coban talun adalah air terjun setin...
2,air terjun,air terjun dolo,"air terjun dolo, mojo, kabupaten kediri, jawa ...",kabupaten kediri,"air terjun dolo, kabupaten kediri, jawa timur",Air Terjun Dolo adalah salah satu tempat wisat...,air terjun dolo adalah salah satu tempat wisat...
3,air terjun,air terjun tumpak sewu,"kawasan air terjun tumpak sewu, jagalan, sidor...",kabupaten malang,"air terjun tumpak sewu, kabupaten malang, jawa...",Air Terjun Tumpak Sewu atau disebut juga Coban...,air terjun tumpak sewu atau disebut juga coban...
4,alun-alun,alun-alun banyuwangi,"jalan tawang alun, biskalan, patemon wetan, ba...",banyuwangi,"alun-alun banyuwangi, banyuwangi, jawa timur",lun-Alun Banyuwangi adalah ruang terbuka publi...,lunalun banyuwangi adalah ruang terbuka publik...
...,...,...,...,...,...,...,...
223,wisata sejarah,benteng van den bosch,"desa pelem, kecamatan ngawi, kabupaten ngawi, ...",ngawi,benteng van den bosch ngawi,Benteng Van den Bosch merupakan peninggalan ko...,benteng van den bosch merupakan peninggalan ko...
224,wisata sejarah,istana gebang,"kelurahan bendogerit, kecamatan sananwetan, ko...",kota blitar,istana gebang blitar,Istana Gebang merupakan rumah masa kecil Presi...,istana gebang merupakan rumah masa kecil presi...
226,wisata kota,kampung biru arema,"kelurahan kiduldalem, kecamatan klojen, kota m...",kota malang,kampung biru arema malang,Kampung Biru Arema merupakan kawasan wisata te...,kampung biru arema merupakan kawasan wisata te...
227,wisata kota,jembatan suramadu,"kelurahan kenjeran, kecamatan bulak, kota sura...",surabaya,jembatan suramadu surabaya,Jembatan Suramadu merupakan jembatan terpanjan...,jembatan suramadu merupakan jembatan terpanjan...


### **Tokenizing**

In [30]:
jj['tokenizing'] = jj['case_folding'].apply(word_tokenize)

/tmp/ipykernel_11716/4164283033.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  jj['tokenizing'] = jj['case_folding'].apply(word_tokenize)


In [31]:
jj

,kategori,nama_wisata,alamat,kota_kabupaten,nama_lengkap,deskripsi_bersih,case_folding,tokenizing
0,air terjun,air terjun coban rondo,"coban rondo, kabupaten malang, jawa timur, jaw...",kabupaten malang,"air terjun coban rondo, kabupaten malang, jawa...",Air Terjun Coban Rondo merupakan salah satu ob...,air terjun coban rondo merupakan salah satu ob...,"[air, terjun, coban, rondo, merupakan, salah, ..."
1,air terjun,air terjun coban talun,"air terjun coban talun, kota batu, jawa timur,...",kota batu,"air terjun coban talun, kota batu, jawa timur",Air Terjun Coban Talun adalah air terjun setin...,air terjun coban talun adalah air terjun setin...,"[air, terjun, coban, talun, adalah, air, terju..."
2,air terjun,air terjun dolo,"air terjun dolo, mojo, kabupaten kediri, jawa ...",kabupaten kediri,"air terjun dolo, kabupaten kediri, jawa timur",Air Terjun Dolo adalah salah satu tempat wisat...,air terjun dolo adalah salah satu tempat wisat...,"[air, terjun, dolo, adalah, salah, satu, tempa..."
3,air terjun,air terjun tumpak sewu,"kawasan air terjun tumpak sewu, jagalan, sidor...",kabupaten malang,"air terjun tumpak sewu, kabupaten malang, jawa...",Air Terjun Tumpak Sewu atau disebut juga Coban...,air terjun tumpak sewu atau disebut juga coban...,"[air, terjun, tumpak, sewu, atau, disebut, jug..."
4,alun-alun,alun-alun banyuwangi,"jalan tawang alun, biskalan, patemon wetan, ba...",banyuwangi,"alun-alun banyuwangi, banyuwangi, jawa timur",lun-Alun Banyuwangi adalah ruang terbuka publi...,lunalun banyuwangi adalah ruang terbuka publik...,"[lunalun, banyuwangi, adalah, ruang, terbuka, ..."
...,...,...,...,...,...,...,...,...
223,wisata sejarah,benteng van den bosch,"desa pelem, kecamatan ngawi, kabupaten ngawi, ...",ngawi,benteng van den bosch ngawi,Benteng Van den Bosch merupakan peninggalan ko...,benteng van den bosch merupakan peninggalan ko...,"[benteng, van, den, bosch, merupakan, peningga..."
224,wisata sejarah,istana gebang,"kelurahan bendogerit, kecamatan sananwetan, ko...",kota blitar,istana gebang blitar,Istana Gebang merupakan rumah masa kecil Presi...,istana gebang merupakan rumah masa kecil presi...,"[istana, gebang, merupakan, rumah, masa, kecil..."
226,wisata kota,kampung biru arema,"kelurahan kiduldalem, kecamatan klojen, kota m...",kota malang,kampung biru arema malang,Kampung Biru Arema merupakan kawasan wisata te...,kampung biru arema merupakan kawasan wisata te...,"[kampung, biru, arema, merupakan, kawasan, wis..."
227,wisata kota,jembatan suramadu,"kelurahan kenjeran, kecamatan bulak, kota sura...",surabaya,jembatan suramadu surabaya,Jembatan Suramadu merupakan jembatan terpanjan...,jembatan suramadu merupakan jembatan terpanjan...,"[jembatan, suramadu, merupakan, jembatan, terp..."


### **Normalization**

Ganti singkatan

In [32]:
normalisasi = {
    "kab.": "kabupaten",
    "kec.": "kecamatan",
    "prov.": "provinsi"
}
def normalize(tokens):
    return [normalisasi.get(word, word) for word in tokens]

jj['normalisasi'] = jj['tokenizing'].apply(normalize)

jj

/tmp/ipykernel_11716/3836164160.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  jj['normalisasi'] = jj['tokenizing'].apply(normalize)


,kategori,nama_wisata,alamat,kota_kabupaten,nama_lengkap,deskripsi_bersih,case_folding,tokenizing,normalisasi
0,air terjun,air terjun coban rondo,"coban rondo, kabupaten malang, jawa timur, jaw...",kabupaten malang,"air terjun coban rondo, kabupaten malang, jawa...",Air Terjun Coban Rondo merupakan salah satu ob...,air terjun coban rondo merupakan salah satu ob...,"[air, terjun, coban, rondo, merupakan, salah, ...","[air, terjun, coban, rondo, merupakan, salah, ..."
1,air terjun,air terjun coban talun,"air terjun coban talun, kota batu, jawa timur,...",kota batu,"air terjun coban talun, kota batu, jawa timur",Air Terjun Coban Talun adalah air terjun setin...,air terjun coban talun adalah air terjun setin...,"[air, terjun, coban, talun, adalah, air, terju...","[air, terjun, coban, talun, adalah, air, terju..."
2,air terjun,air terjun dolo,"air terjun dolo, mojo, kabupaten kediri, jawa ...",kabupaten kediri,"air terjun dolo, kabupaten kediri, jawa timur",Air Terjun Dolo adalah salah satu tempat wisat...,air terjun dolo adalah salah satu tempat wisat...,"[air, terjun, dolo, adalah, salah, satu, tempa...","[air, terjun, dolo, adalah, salah, satu, tempa..."
3,air terjun,air terjun tumpak sewu,"kawasan air terjun tumpak sewu, jagalan, sidor...",kabupaten malang,"air terjun tumpak sewu, kabupaten malang, jawa...",Air Terjun Tumpak Sewu atau disebut juga Coban...,air terjun tumpak sewu atau disebut juga coban...,"[air, terjun, tumpak, sewu, atau, disebut, jug...","[air, terjun, tumpak, sewu, atau, disebut, jug..."
4,alun-alun,alun-alun banyuwangi,"jalan tawang alun, biskalan, patemon wetan, ba...",banyuwangi,"alun-alun banyuwangi, banyuwangi, jawa timur",lun-Alun Banyuwangi adalah ruang terbuka publi...,lunalun banyuwangi adalah ruang terbuka publik...,"[lunalun, banyuwangi, adalah, ruang, terbuka, ...","[lunalun, banyuwangi, adalah, ruang, terbuka, ..."
...,...,...,...,...,...,...,...,...,...
223,wisata sejarah,benteng van den bosch,"desa pelem, kecamatan ngawi, kabupaten ngawi, ...",ngawi,benteng van den bosch ngawi,Benteng Van den Bosch merupakan peninggalan ko...,benteng van den bosch merupakan peninggalan ko...,"[benteng, van, den, bosch, merupakan, peningga...","[benteng, van, den, bosch, merupakan, peningga..."
224,wisata sejarah,istana gebang,"kelurahan bendogerit, kecamatan sananwetan, ko...",kota blitar,istana gebang blitar,Istana Gebang merupakan rumah masa kecil Presi...,istana gebang merupakan rumah masa kecil presi...,"[istana, gebang, merupakan, rumah, masa, kecil...","[istana, gebang, merupakan, rumah, masa, kecil..."
226,wisata kota,kampung biru arema,"kelurahan kiduldalem, kecamatan klojen, kota m...",kota malang,kampung biru arema malang,Kampung Biru Arema merupakan kawasan wisata te...,kampung biru arema merupakan kawasan wisata te...,"[kampung, biru, arema, merupakan, kawasan, wis...","[kampung, biru, arema, merupakan, kawasan, wis..."
227,wisata kota,jembatan suramadu,"kelurahan kenjeran, kecamatan bulak, kota sura...",surabaya,jembatan suramadu surabaya,Jembatan Suramadu merupakan jembatan terpanjan...,jembatan suramadu merupakan jembatan terpanjan...,"[jembatan, suramadu, merupakan, jembatan, terp...","[jembatan, suramadu, merupakan, jembatan, terp..."


### **Filtering**

Hapus stopword

In [33]:
stop_words = set(stopwords.words('indonesian'))

def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]

jj['filtering'] = jj['normalisasi'].apply(remove_stopwords)

jj

/tmp/ipykernel_11716/3647231475.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  jj['filtering'] = jj['normalisasi'].apply(remove_stopwords)


,kategori,nama_wisata,alamat,kota_kabupaten,nama_lengkap,deskripsi_bersih,case_folding,tokenizing,normalisasi,filtering
0,air terjun,air terjun coban rondo,"coban rondo, kabupaten malang, jawa timur, jaw...",kabupaten malang,"air terjun coban rondo, kabupaten malang, jawa...",Air Terjun Coban Rondo merupakan salah satu ob...,air terjun coban rondo merupakan salah satu ob...,"[air, terjun, coban, rondo, merupakan, salah, ...","[air, terjun, coban, rondo, merupakan, salah, ...","[air, terjun, coban, rondo, salah, objek, wisa..."
1,air terjun,air terjun coban talun,"air terjun coban talun, kota batu, jawa timur,...",kota batu,"air terjun coban talun, kota batu, jawa timur",Air Terjun Coban Talun adalah air terjun setin...,air terjun coban talun adalah air terjun setin...,"[air, terjun, coban, talun, adalah, air, terju...","[air, terjun, coban, talun, adalah, air, terju...","[air, terjun, coban, talun, air, terjun, meter..."
2,air terjun,air terjun dolo,"air terjun dolo, mojo, kabupaten kediri, jawa ...",kabupaten kediri,"air terjun dolo, kabupaten kediri, jawa timur",Air Terjun Dolo adalah salah satu tempat wisat...,air terjun dolo adalah salah satu tempat wisat...,"[air, terjun, dolo, adalah, salah, satu, tempa...","[air, terjun, dolo, adalah, salah, satu, tempa...","[air, terjun, dolo, salah, wisata, air, terjun..."
3,air terjun,air terjun tumpak sewu,"kawasan air terjun tumpak sewu, jagalan, sidor...",kabupaten malang,"air terjun tumpak sewu, kabupaten malang, jawa...",Air Terjun Tumpak Sewu atau disebut juga Coban...,air terjun tumpak sewu atau disebut juga coban...,"[air, terjun, tumpak, sewu, atau, disebut, jug...","[air, terjun, tumpak, sewu, atau, disebut, jug...","[air, terjun, tumpak, sewu, coban, sewu, air, ..."
4,alun-alun,alun-alun banyuwangi,"jalan tawang alun, biskalan, patemon wetan, ba...",banyuwangi,"alun-alun banyuwangi, banyuwangi, jawa timur",lun-Alun Banyuwangi adalah ruang terbuka publi...,lunalun banyuwangi adalah ruang terbuka publik...,"[lunalun, banyuwangi, adalah, ruang, terbuka, ...","[lunalun, banyuwangi, adalah, ruang, terbuka, ...","[lunalun, banyuwangi, ruang, terbuka, publik, ..."
...,...,...,...,...,...,...,...,...,...,...
223,wisata sejarah,benteng van den bosch,"desa pelem, kecamatan ngawi, kabupaten ngawi, ...",ngawi,benteng van den bosch ngawi,Benteng Van den Bosch merupakan peninggalan ko...,benteng van den bosch merupakan peninggalan ko...,"[benteng, van, den, bosch, merupakan, peningga...","[benteng, van, den, bosch, merupakan, peningga...","[benteng, van, den, bosch, peninggalan, koloni..."
224,wisata sejarah,istana gebang,"kelurahan bendogerit, kecamatan sananwetan, ko...",kota blitar,istana gebang blitar,Istana Gebang merupakan rumah masa kecil Presi...,istana gebang merupakan rumah masa kecil presi...,"[istana, gebang, merupakan, rumah, masa, kecil...","[istana, gebang, merupakan, rumah, masa, kecil...","[istana, gebang, rumah, presiden, soekarno, di..."
226,wisata kota,kampung biru arema,"kelurahan kiduldalem, kecamatan klojen, kota m...",kota malang,kampung biru arema malang,Kampung Biru Arema merupakan kawasan wisata te...,kampung biru arema merupakan kawasan wisata te...,"[kampung, biru, arema, merupakan, kawasan, wis...","[kampung, biru, arema, merupakan, kawasan, wis...","[kampung, biru, arema, kawasan, wisata, temati..."
227,wisata kota,jembatan suramadu,"kelurahan kenjeran, kecamatan bulak, kota sura...",surabaya,jembatan suramadu surabaya,Jembatan Suramadu merupakan jembatan terpanjan...,jembatan suramadu merupakan jembatan terpanjan...,"[jembatan, suramadu, merupakan, jembatan, terp...","[jembatan, suramadu, merupakan, jembatan, terp...","[jembatan, suramadu, jembatan, terpanjang, ind..."


### **Stemming**

In [34]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()

def stemming(tokens):
    return [stemmer.stem(word) for word in tokens]

jj['stemming'] = jj['filtering'].apply(stemming)

jj

/tmp/ipykernel_11716/1262886401.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  jj['stemming'] = jj['filtering'].apply(stemming)


,kategori,nama_wisata,alamat,kota_kabupaten,nama_lengkap,deskripsi_bersih,case_folding,tokenizing,normalisasi,filtering,stemming
0,air terjun,air terjun coban rondo,"coban rondo, kabupaten malang, jawa timur, jaw...",kabupaten malang,"air terjun coban rondo, kabupaten malang, jawa...",Air Terjun Coban Rondo merupakan salah satu ob...,air terjun coban rondo merupakan salah satu ob...,"[air, terjun, coban, rondo, merupakan, salah, ...","[air, terjun, coban, rondo, merupakan, salah, ...","[air, terjun, coban, rondo, salah, objek, wisa...","[air, terjun, coban, rondo, salah, objek, wisa..."
1,air terjun,air terjun coban talun,"air terjun coban talun, kota batu, jawa timur,...",kota batu,"air terjun coban talun, kota batu, jawa timur",Air Terjun Coban Talun adalah air terjun setin...,air terjun coban talun adalah air terjun setin...,"[air, terjun, coban, talun, adalah, air, terju...","[air, terjun, coban, talun, adalah, air, terju...","[air, terjun, coban, talun, air, terjun, meter...","[air, terjun, coban, talun, air, terjun, meter..."
2,air terjun,air terjun dolo,"air terjun dolo, mojo, kabupaten kediri, jawa ...",kabupaten kediri,"air terjun dolo, kabupaten kediri, jawa timur",Air Terjun Dolo adalah salah satu tempat wisat...,air terjun dolo adalah salah satu tempat wisat...,"[air, terjun, dolo, adalah, salah, satu, tempa...","[air, terjun, dolo, adalah, salah, satu, tempa...","[air, terjun, dolo, salah, wisata, air, terjun...","[air, terjun, dolo, salah, wisata, air, terjun..."
3,air terjun,air terjun tumpak sewu,"kawasan air terjun tumpak sewu, jagalan, sidor...",kabupaten malang,"air terjun tumpak sewu, kabupaten malang, jawa...",Air Terjun Tumpak Sewu atau disebut juga Coban...,air terjun tumpak sewu atau disebut juga coban...,"[air, terjun, tumpak, sewu, atau, disebut, jug...","[air, terjun, tumpak, sewu, atau, disebut, jug...","[air, terjun, tumpak, sewu, coban, sewu, air, ...","[air, terjun, tumpak, sewu, coban, sewu, air, ..."
4,alun-alun,alun-alun banyuwangi,"jalan tawang alun, biskalan, patemon wetan, ba...",banyuwangi,"alun-alun banyuwangi, banyuwangi, jawa timur",lun-Alun Banyuwangi adalah ruang terbuka publi...,lunalun banyuwangi adalah ruang terbuka publik...,"[lunalun, banyuwangi, adalah, ruang, terbuka, ...","[lunalun, banyuwangi, adalah, ruang, terbuka, ...","[lunalun, banyuwangi, ruang, terbuka, publik, ...","[lunalun, banyuwangi, ruang, buka, publik, let..."
...,...,...,...,...,...,...,...,...,...,...,...
223,wisata sejarah,benteng van den bosch,"desa pelem, kecamatan ngawi, kabupaten ngawi, ...",ngawi,benteng van den bosch ngawi,Benteng Van den Bosch merupakan peninggalan ko...,benteng van den bosch merupakan peninggalan ko...,"[benteng, van, den, bosch, merupakan, peningga...","[benteng, van, den, bosch, merupakan, peningga...","[benteng, van, den, bosch, peninggalan, koloni...","[benteng, van, den, bosch, tinggal, kolonial, ..."
224,wisata sejarah,istana gebang,"kelurahan bendogerit, kecamatan sananwetan, ko...",kota blitar,istana gebang blitar,Istana Gebang merupakan rumah masa kecil Presi...,istana gebang merupakan rumah masa kecil presi...,"[istana, gebang, merupakan, rumah, masa, kecil...","[istana, gebang, merupakan, rumah, masa, kecil...","[istana, gebang, rumah, presiden, soekarno, di...","[istana, gebang, rumah, presiden, soekarno, ja..."
226,wisata kota,kampung biru arema,"kelurahan kiduldalem, kecamatan klojen, kota m...",kota malang,kampung biru arema malang,Kampung Biru Arema merupakan kawasan wisata te...,kampung biru arema merupakan kawasan wisata te...,"[kampung, biru, arema, merupakan, kawasan, wis...","[kampung, biru, arema, merupakan, kawasan, wis...","[kampung, biru, arema, kawasan, wisata, temati...","[kampung, biru, arema, kawasan, wisata, temati..."
227,wisata kota,jembatan suramadu,"kelurahan kenjeran, kecamatan bulak, kota sura...",surabaya,jembatan suramadu surabaya,Jembatan Suramadu merupakan jembatan terpanjan...,jembatan suramadu merupakan jembatan terpanjan...,"[jembatan, surama

## **Vectorization**

TF-IDF

In [35]:
desk_lengkap = jj['nama_lengkap'].astype(str).str.lower()
jj_ready = desk_lengkap + " " + jj['stemming'].apply(lambda x: ' '.join(x) if isinstance(x, list) else str(x))
print("Hasil Penggabungan (Nama + Deskripsi Stemmed):")
print(jj_ready.head())

Hasil Penggabungan (Nama + Deskripsi Stemmed):
0    air terjun coban rondo, kabupaten malang, jawa...
1    air terjun coban talun, kota batu, jawa timur ...
2    air terjun dolo, kabupaten kediri, jawa timur ...
3    air terjun tumpak sewu, kabupaten malang, jawa...
4    alun-alun banyuwangi, banyuwangi, jawa timur l...
dtype: object


In [36]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(jj_ready)
print(f"Dimensi Matriks: {tfidf_matrix.shape}")

Dimensi Matriks: (219, 1473)


In [37]:
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=vectorizer.get_feature_names_out()
)

In [38]:
print(tfidf_df)

     abad  abadi     acara  ada  administrasi  adopsi  adventure  aga  agam  \
0     0.0    0.0  0.000000  0.0           0.0     0.0        0.0  0.0   0.0   
1     0.0    0.0  0.000000  0.0           0.0     0.0        0.0  0.0   0.0   
2     0.0    0.0  0.000000  0.0           0.0     0.0        0.0  0.0   0.0   
3     0.0    0.0  0.000000  0.0           0.0     0.0        0.0  0.0   0.0   
4     0.0    0.0  0.195152  0.0           0.0     0.0        0.0  0.0   0.0   
..    ...    ...       ...  ...           ...     ...        ...  ...   ...   
214   0.0    0.0  0.000000  0.0           0.0     0.0        0.0  0.0   0.0   
215   0.0    0.0  0.000000  0.0           0.0     0.0        0.0  0.0   0.0   
216   0.0    0.0  0.000000  0.0           0.0     0.0        0.0  0.0   0.0   
217   0.0    0.0  0.000000  0.0           0.0     0.0        0.0  0.0   0.0   
218   0.0    0.0  0.000000  0.0           0.0     0.0        0.0  0.0   0.0   

     agama  ...  wringin  xii  yakin  yani  yesus  

Dataset akhir punya 219 dokumen wisata dan 1473 fitur kata setelah preprocessing (case folding, tokenizing, stopword removal, stemming, dll).